# Risk Classification — Fine-tuning DistilBERT

**Strategy:**
- Serialize numeric supply chain features as natural language
- Concatenate with matched news headline
- Fine-tune `distilbert-base-uncased` for 3-class classification
- Classes: `Low Risk`, `Moderate Risk`, `High Risk`

**Input pipeline:**
```
merged_supply_chain_with_news.csv
        ↓
Feature serialization → text prompt
        ↓
DistilBERT tokenizer
        ↓
Fine-tune classifier head
        ↓
Evaluate + save model
```

## Step 0: Install Dependencies

In [1]:
# Run once — skip if already installed
!pip install transformers datasets scikit-learn torch accelerate -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


## Step 1: Load the Merged Dataset

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("merged_supply_chain_with_news.csv")

print(f"Shape: {df.shape}")
print(f"Target distribution:")
print(df['risk_classification'].value_counts())

Shape: (32065, 29)
Target distribution:
risk_classification
High Risk        23944
Moderate Risk     5011
Low Risk          3110
Name: count, dtype: int64


## Step 2: Label Encoding

Map `risk_classification` string labels to integer IDs.

In [3]:
# Define label mapping (alphabetical → consistent order)
label2id = {
    'Low Risk':      0,
    'Moderate Risk': 1,
    'High Risk':     2,
}
id2label = {v: k for k, v in label2id.items()}

df['label'] = df['risk_classification'].map(label2id)

# Drop rows where label mapping failed (unexpected class values)
before = len(df)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
print(f"Dropped {before - len(df)} rows with unknown risk_classification values.")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")

Dropped 0 rows with unknown risk_classification values.
Label distribution: {2: 23944, 1: 5011, 0: 3110}


## Step 3: Feature Serialization

Convert numeric columns into a readable sentence, then append the news headline.

**Why this works for LLMs:** DistilBERT was trained on natural language. Describing numbers as phrases ("fuel consumption rate is 5.13") gives the model context it can attend to — much better than raw floats.

In [4]:
# Numeric feature columns to serialize
NUMERIC_FEATURES = [
    'fuel_consumption_rate',
    'eta_variation_hours',
    'traffic_congestion_level',
    'warehouse_inventory_level',
    'loading_unloading_time',
    'handling_equipment_availability',
    'weather_condition_severity',
    'port_congestion_level',
    'shipping_costs',
    'supplier_reliability_score',
    'lead_time_days',
    'route_risk_level',
    'customs_clearance_time',
    'driver_behavior_score',
    'fatigue_monitoring_score',
    'disruption_likelihood_score',
    'delay_probability',
    'delivery_time_deviation',
]

# Only keep features that exist in the dataframe
NUMERIC_FEATURES = [f for f in NUMERIC_FEATURES if f in df.columns]
print(f"Using {len(NUMERIC_FEATURES)} numeric features.")

def serialize_row(row):
    """
    Convert a row into a natural-language prompt:
    'fuel consumption rate is 5.13. eta variation hours is 4.99. ... 
     News: [headline]'
    """
    parts = []
    for col in NUMERIC_FEATURES:
        val = row[col]
        label_text = col.replace('_', ' ')
        parts.append(f"{label_text} is {round(float(val), 3)}")
    
    feature_text = ". ".join(parts)
    headline = str(row['headline']) if pd.notna(row['headline']) else 'No relevant news'
    
    return f"{feature_text}. News: {headline}"

print("Serializing rows (this may take ~1 min for large datasets)...")
df['input_text'] = df.apply(serialize_row, axis=1)

print(f"Done. Example input text:")
print(df['input_text'].iloc[0][:400], "...")

Using 18 numeric features.
Serializing rows (this may take ~1 min for large datasets)...
Done. Example input text:
fuel consumption rate is 5.137. eta variation hours is 4.998. traffic congestion level is 5.928. warehouse inventory level is 985.717. loading unloading time is 4.951. handling equipment availability is 0.481. weather condition severity is 0.359. port congestion level is 4.289. shipping costs is 456.504. supplier reliability score is 0.986. lead time days is 2.128. route risk level is 1.182. custo ...


## Step 4: TF-IDF Analysis (Feature Importance Check)

Before training, we use TF-IDF on the serialized text to verify that the most discriminative terms align with supply chain domain knowledge.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Sample for TF-IDF analysis (full set can be slow)
sample_df = df.sample(min(10000, len(df)), random_state=42)

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(sample_df['input_text'])
y_tfidf = sample_df['label']

# Quick logistic regression baseline
X_tr, X_te, y_tr, y_te = train_test_split(X_tfidf, y_tfidf, test_size=0.2, random_state=42, stratify=y_tfidf)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr, y_tr)
y_pred = lr.predict(X_te)

print("=== TF-IDF + Logistic Regression Baseline ===")
print(classification_report(y_te, y_pred, target_names=list(label2id.keys())))
print("(This is our baseline — DistilBERT should beat this significantly)")

=== TF-IDF + Logistic Regression Baseline ===
               precision    recall  f1-score   support

     Low Risk       0.00      0.00      0.00       193
Moderate Risk       0.50      0.00      0.01       311
    High Risk       0.75      1.00      0.86      1496

     accuracy                           0.75      2000
    macro avg       0.42      0.33      0.29      2000
 weighted avg       0.64      0.75      0.64      2000

(This is our baseline — DistilBERT should beat this significantly)


In [6]:
# Top TF-IDF terms per class
feature_names = np.array(tfidf.get_feature_names_out())

for class_id, class_name in id2label.items():
    coef = lr.coef_[class_id]
    top_idx = np.argsort(coef)[-10:][::-1]
    top_terms = feature_names[top_idx]
    print(f"\nTop terms for '{class_name}':")
    print(", ".join(top_terms))


Top terms for 'Low Risk':
05, 022, 115, 036, 152, 122, 015, is 015, is 022, 141

Top terms for 'Moderate Risk':
is, 501, 031, is 501, 51, 913, 04, is driver, is 94, 959

Top terms for 'High Risk':
is delay, 999 delay, 998 delay, 997 delay, 875, is 996, 993, is 995, 996, is 998


## Step 5: Train/Validation/Test Split

In [7]:
from sklearn.model_selection import train_test_split

# For faster fine-tuning, optionally sample (remove cap for full training)
MAX_SAMPLES = None  # set to None to use full dataset

model_df = df[['input_text', 'label']].dropna()
if MAX_SAMPLES and len(model_df) > MAX_SAMPLES:
    model_df = model_df.sample(MAX_SAMPLES, random_state=42)
    print(f"Sampled to {MAX_SAMPLES} rows for training.")

train_df, temp_df = train_test_split(model_df, test_size=0.2, random_state=42, stratify=model_df['label'])
val_df, test_df   = train_test_split(temp_df,  test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Label distribution in train: {train_df['label'].value_counts().to_dict()}")

Train: 25,652 | Val: 3,206 | Test: 3,207
Label distribution in train: {2: 19155, 1: 4009, 0: 2488}


In [10]:
import sys
!{sys.executable} -m pip install datasets

  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ------------------- -------------------- 41.0/82.2 kB 2.0 MB/s eta 0:00:01
     --------------------------------- ---- 71.7/82.2 kB 991.0 kB/s eta 0:00:01
     ---------------------------------------- 82.2/82.2 kB 1.2 MB/s eta 0:00:00
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
   ---------------------------------------- 0.0/144.5 kB ? eta -:--:--
   ----------- ---------------------------- 41.0/144.5 kB 1.9 MB/s eta 0:00:01
   ------------------------------- -------- 112.6/1


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 6: Tokenize with DistilBERT

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 512  # DistilBERT max token length

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['input_text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
    )

# Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

# Set format for PyTorch
cols = ['input_ids', 'attention_mask', 'label']
train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)

print("Tokenization complete.")
print(f"Example token length: {len(train_ds[0]['input_ids'])}")

Map: 100%|██████████| 3207/3207 [00:00<00:00, 4758.15 examples/s]

Tokenization complete.
Example token length: 512


In [12]:
import sys
print(sys.executable)

c:\Users\daksh\AppData\Local\Programs\Python\Python311\python.exe


In [22]:
import sys
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import sys
!{sys.executable} -m pip install --upgrade transformers accelerate

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 7: Load DistilBERT for Sequence Classification

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5882.20it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     66,955,779
Trainable parameters: 66,955,779


## Step 8: Training Configuration

In [14]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1_weighted': f1}

training_args = TrainingArguments(
    output_dir='./distilbert_risk_model',

    # Core hyperparameters
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    # Evaluation
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',

    # Logging
    logging_steps=50,
    logging_dir='./logs',

    # Reproducibility
    seed=42,

    # Speed (enable if GPU available)
    fp16=False,  # set True if using GPU with CUDA
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Trainer ready.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer ready.


## Step 9: Fine-tune!

> **GPU strongly recommended.** On CPU, expect ~2–3 hrs for 20k samples × 3 epochs.  
> On a T4/V100 GPU (Google Colab free tier), expect ~15–25 min.

In [13]:
import sys
!{sys.executable} -m pip uninstall torch torchvision torchaudio -y

In [15]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 5.3 MB/s eta 0:07:44
     ---------------------------------------- 0.0/2.4 GB 9.7 MB/s eta 0:04:13
     ---------------------------------------- 0.0/2.4 GB 18.7 MB/s eta 0:02:11
     ---------------------------------------- 0.0/2.4 GB 20.5 MB/s eta 0:02:00
     ---------------------------------------- 0.0/2.4 GB 21.0 MB/s eta 0:01:57
     ---------------------------------------- 0.0/2.4 GB 21.0 MB/s eta 0:01:57
     ---------------------------------------- 0.0/2.4 GB 21.0 MB/s eta 0:01:57
     ---------------------------------------- 0.0/2.4 GB 21.0 MB/s eta 0:01:57
     ---------------------------------------- 0.0/2.4 GB 21.0 MB/s eta 0:01:57
     ---------------------------------------- 0.0/2.4 GB 21.0 MB/s eta 0:01:57
     ---------------------------------------- 0.0/2.4 GB 21.0 MB/s eta 0:01:57
   

ERROR: Exception:
Traceback (most recent call last):
  File "c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\http\client.py", line 473, in read
    s = self.fp.read

In [11]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.11.0+cpu
None


In [10]:
import sys
print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
c:\Users\daksh\AppData\Local\Programs\Python\Python311\python.exe


In [9]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())


if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

CUDA available: False
Device count: 0


In [2]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device.upper()}")

train_result = trainer.train()

print("\n=== Training Complete ===")
print(f"Total steps: {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")

Training on: CPU


NameError: name 'trainer' is not defined

## Step 10: Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions
pred_output = trainer.predict(test_ds)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print("=== Test Set Results ===")
print(classification_report(
    y_true, y_pred,
    target_names=[id2label[i] for i in range(3)]
))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[id2label[i] for i in range(3)],
    yticklabels=[id2label[i] for i in range(3)]
)
plt.title('Confusion Matrix — Risk Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Saved confusion_matrix.png")

## Step 11: Save the Fine-tuned Model

In [ ]:
SAVE_DIR = './distilbert_risk_model_final'

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model and tokenizer saved to: {SAVE_DIR}")

## Step 12: Inference — Predict on New Rows

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    truncation=True,
    max_length=MAX_LEN,
)

# Example: predict on a single new row from the test set
sample_text = test_df.iloc[0]['input_text']
true_label  = id2label[test_df.iloc[0]['label']]

result = classifier(sample_text[:512])

print(f"Input (truncated): {sample_text[:300]}...")
print(f"\nTrue label:      {true_label}")
print(f"Predicted label: {result[0]['label']}")
print(f"Confidence:      {result[0]['score']:.2%}")

In [ ]:
# Batch inference on 10 test samples
samples = test_df.head(10)
texts = [t[:512] for t in samples['input_text'].tolist()]
results = classifier(texts)

comparison = pd.DataFrame({
    'true_label':      [id2label[l] for l in samples['label'].tolist()],
    'predicted_label': [r['label'] for r in results],
    'confidence':      [f"{r['score']:.2%}" for r in results],
})
comparison['correct'] = comparison['true_label'] == comparison['predicted_label']
print(comparison.to_string(index=False))